In [ ]:
from pathlib import Path
import pandas as pd 


In [ ]:
print(Path.cwd())

In [ ]:
project_root = (
    Path.cwd().parent 
)
print(project_root)

data_path = project_root.joinpath('data','raw') 
print(data_path) 


In [ ]:
orders = pd.read_csv(data_path.joinpath('olist_orders_dataset.csv')) 

order_items = pd.read_csv(data_path.joinpath('olist_order_items_dataset.csv')) 

customers = pd.read_csv(data_path.joinpath('olist_customers_dataset.csv')) 



In [ ]:
# orders.head() 
# order_items.head() 
customers.head(10) 

In [ ]:
# .shape 返回(行数, 列数)
print("订单号:",orders.shape)

print("商品明细表:",order_items.shape) 

print("客户表:",customers.shape)

In [ ]:
orders.info() 

In [ ]:
order_items.info() 

In [ ]:
customers.info() 

In [ ]:
print(orders.columns.tolist()) 

print(order_items.columns.tolist()) 

print(customers.columns.tolist())


In [ ]:
# ex:

# print(len(orders.columns))
# print(len(order_items.columns)) 
# print(len(customers.columns)) 

# print(orders.columns[0]) 
# print(orders['order_id']) 

# print(type(orders.columns)) 


In [ ]:
print(orders.isna().head()) 
print(orders.isna().sum())



In [ ]:
print(order_items.isna().sum()) 
print(customers.isna().sum()) 

In [ ]:
orders.isna().mean() * 100 

In [ ]:
missing_orders = pd.DataFrame({
    '缺失数量': orders.isna().sum(), 
    '缺失比例(%)': orders.isna().mean() * 100 
}) 

missing_orders

In [ ]:
missing_orders.sort_values(
    by = '缺失比例(%)', 
    ascending=False 
)

## 调查缺失值原因

* order_delivered_customer_date 
* order_delivered_carrier_date 
* order_approved_at

In [ ]:
orders['order_status'].value_counts()

In [ ]:
missing_delivery = orders[
    orders['order_delivered_customer_date'].isna() 
]
missing_delivery 

In [ ]:
missing_delivery['order_status'].value_counts() 

In [ ]:
# 筛选8个status为delivery,但是order_delivered_customer_date缺失的数据

abnormal = missing_delivery[
    missing_delivery['order_status'] == 'delivered'
]
abnormal[[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_estimated_delivery_date"
]]

### 已送达订单的实际送达时间缺失问题

**1. 问题发现**

在 96,478 笔状态为 `delivered`（已送达）的订单中，发现 **8 笔订单缺失实际送达时间** `order_delivered_customer_date`，约占已送达订单的 0.0083%。

**2. 进一步检查**

- 8 笔订单均存在下单时间、获批时间和预计送达日期。
- 其中 7 笔有交付物流时间，另外 1 笔也缺失交付物流时间。
- 订单状态显示为已送达，但无法确定实际送达的具体日期。
- 目前无法判断缺失原因，可能存在物流记录不完整或数据更新不一致等情况。

**3. 对延迟预测的影响**

本项目需要通过比较实际送达日期 `order_delivered_customer_date` 与预计送达日期 `order_estimated_delivery_date`，判断订单是否延迟：

- 实际送达日期晚于预计送达日期：延迟（标签为 1）。
- 实际送达日期不晚于预计送达日期：未延迟（标签为 0）。

由于这 8 笔订单缺失实际送达时间，**无法确定它们是否发生延迟，因此不能为它们生成可靠的预测标签**。

**4. 后续处理方案**

- 数据探索阶段保留这 8 笔订单，并记录缺失情况。
- 后续构建延迟预测数据集时，将这 8 笔订单排除在有确定标签的建模样本之外。
- 不直接将缺失实际送达时间的订单标记为“未延迟”。
- 不使用预计送达日期或其他日期直接填补实际送达日期，以免引入错误标签。

**结论：** 这 8 笔订单虽然有订单状态和部分物流信息，但由于缺少实际送达时间，无法准确判断是否延迟。它们可以保留在原始数据中用于数据质量分析，但不适合直接用于需要明确延迟标签的监督学习训练与评估。

In [ ]:
# order_status是取消但是有送达时间
canceled_delivered = orders[
    (orders['order_status']=='canceled') & 
    (orders['order_delivered_customer_date'].notna()) 
]
canceled_delivered[[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]] 

### 已取消但存在实际送达时间的订单分析

在 625 笔状态为 `canceled` 的订单中，发现 6 笔仍然存在实际送达时间记录。

进一步检查发现：

- 6 笔订单均具有下单、获批、交付物流及实际送达时间。
- 其中 5 笔早于预计送达日期完成配送，1 笔晚于预计送达日期。
- 订单虽然记录了完整的配送流程，但最终状态却为 `canceled`。

**可能原因：** 这些订单可能涉及收货后的退货、售后取消，或者订单状态更新与物流信息不一致。但现有数据不包含足够的退货、退款信息，无法确认具体原因。

**后续处理：** 数据质量检查阶段保留这些订单并记录异常。构建第一版延迟预测数据集时，暂时排除这 6 笔已取消订单，以保证建模样本的定义一致。

## 重复值检查 

In [ ]:
print(orders.duplicated().sum()) 
print(order_items.duplicated().sum()) 
print(customers.duplicated().sum()) 


In [ ]:
print(orders['order_id'].duplicated().sum()) 
print(customers['customer_id'].duplicated().sum()) 
print(customers['customer_unique_id'].duplicated().sum()) 
print(order_items['order_id'].duplicated().sum()) 
print(order_items.duplicated(
    subset=['order_id', 'order_item_id'] 
).sum()) 

## 3. 重复值检查

### 3.1 检查方法

使用 Pandas 的 `duplicated()` 方法检查重复记录：

- `df.duplicated().sum()`：统计整行完全重复的记录数量。
- `df["列名"].duplicated().sum()`：统计某个字段重复出现的记录数量（不包括第一次出现的记录）。
- `df.duplicated(subset=["列1", "列2"]).sum()`：根据多个字段的组合检查重复记录。
- `df["列名"].is_unique`：判断某一列的值是否全部唯一。

### 3.2 订单表（Orders）

订单表中，每行代表一笔订单，`order_id` 是订单的唯一标识。

检查结果表明，订单表不存在重复的 `order_id`，符合一笔订单对应一条记录的数据结构。

### 3.3 客户表（Customers）

客户表通过 `customer_id` 区分不同的客户订单记录。

- `customer_id` 应当唯一，以便与订单表正确关联。
- `customer_unique_id` 可以重复，因为同一个客户可能存在多笔订单。

检查结果符合上述关系，因此 `customer_unique_id` 的重复属于正常的业务现象，不需要去重。

### 3.4 商品明细表（Order Items）

商品明细表中，每行代表一笔订单中的一条商品明细。

- `order_id` 可以重复，因为一笔订单可能包含多条商品明细。
- `order_item_id` 是同一订单内部的明细序号。
- `order_id` 和 `order_item_id` 的组合应当唯一。

检查结果表明，商品明细表的重复情况符合一笔订单对应多条商品明细的业务关系。

### 3.5 结论

**三张表的重复情况总体符合预期的数据结构，暂未发现需要直接删除的重复记录。**

需要注意：

- 字段重复不一定意味着数据错误。
- 判断是否需要去重，应当结合每张表的数据粒度和关键字段。
- 后续进行多表关联时，应检查关联后订单数量是否发生异常增加，避免重复匹配导致统计错误。

## 日期检查

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], 
    errors='coerce' 
) 
orders['order_purchase_timestamp'].dtype

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# for col in date_columns: 
#     orders[col] = pd.to_datetime(
#         orders[col], 
#         errors="coerce" 
#     )

# 记录转换之前的缺失值数量
missing_before = orders[date_columns].isna().sum()


orders[date_columns] = orders[date_columns].apply(
    pd.to_datetime, 
    errors="coerce"
)
orders[date_columns].dtypes


# 转换后的缺失值数量
missing_after = orders[date_columns].isna().sum()
# 新增的缺失数量
new_missing = missing_after - missing_before


new_missing


#### 检查时间是否合理 

In [ ]:
abnormal_approval = orders[
    orders['order_purchase_timestamp'] > orders['order_approved_at'] 
]

print("获批早于下单时间的数量:", len(abnormal_approval)) 

abnormal_approval[[
    'order_id', 
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_status'
]].head(10)

In [ ]:
abnormal_carrier = orders[
    orders['order_approved_at'] > orders['order_delivered_carrier_date'] 
]

print('交付物流早于获批的订单数量:', len(abnormal_carrier)) 

abnormal_carrier[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date"
]].head(10) 

In [ ]:
abnormal_delivery = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
]

print("实际送达早于交付物流的订单数量:", len(abnormal_delivery))

abnormal_delivery[[
    "order_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]].head(10)


In [ ]:
date_summary = pd.DataFrame({
    '异常类型': [
        "实际送达早于物流交接",
        "物流交接早于订单获批",
        "订单获批早于下单"      
    ],
    '异常数量': [
        len(abnormal_delivery), 
        len(abnormal_carrier), 
        len(abnormal_approval) 
    ]
})

date_summary

In [ ]:
date_summary['异常比例(%)'] = (
    date_summary['异常数量']/len(orders) * 100
)
date_summary

## 4. 日期异常检查

### 4.1 日期类型转换

原始订单表中的时间字段为字符串类型（`str`），为了进行日期比较，使用 `pd.to_datetime()` 将其转换为 `datetime` 类型。

使用 `errors="coerce"` 将无法解析的日期转换为 `NaT`。

检查转换前后的缺失值数量，以判断日期转换是否产生了新的缺失值。

- **日期转换新增缺失值：** 待填写

### 4.2 时间顺序异常检查

根据正常的订单履约流程：

**客户下单 → 订单获批 → 交付物流 → 客户收货**

本次主要检查以下三类时间顺序异常：

| 异常类型 | 判断条件 | 异常数量 |
|---|---|---:|
| 实际送达早于物流交接 | `delivered_customer_date < delivered_carrier_date` | 待填写 |
| 物流交接早于订单获批 | `delivered_carrier_date < approved_at` | 待填写 |
| 订单获批早于下单 | `approved_at < purchase_timestamp` | 待填写 |

注：表中判断条件为字段名称的简写，实际代码使用订单表中的完整字段名。

### 4.3 异常原因分析

以上时间顺序通常应当符合订单履约的先后关系。

如果发现时间倒序，可能存在以下原因：

- 系统时间记录或数据同步存在偏差。
- 不同业务环节的时间戳记录规则不同。
- 部分订单的时间字段存在数据质量问题。

**注意：** 目前无法仅凭时间倒序确定具体原因，需要结合异常时间差和其他字段进一步调查。

### 4.4 后续处理建议

- 暂时保留异常记录，不直接删除或修改。
- 记录各类异常的数量和比例。
- 对异常记录进一步检查时间差，区分相差几秒、几分钟或数天的情况。
- 后续建模时，根据预测时点、标签定义以及异常字段是否影响分析结果，决定是否排除相关订单。

### 4.5 初步结论

本次完成了订单日期字段的类型转换，并对订单履约过程中的三种时间顺序进行了检查。

具体异常数量及其影响需要根据实际运行结果补充。时间倒序记录应作为需要进一步核实的数据质量问题，而不能直接认定为无效订单。